In [20]:
from llama_index.core.bridge.pydantic import BaseModel, Field
from llama_index.llms.ollama import Ollama
import json

In [21]:
class Guess(BaseModel):
    answer: str = Field(description="The answer to the clue, fitting the specified length and any known letters. The answer should be in uppercase and should not contain spaces or punctuation.")
    confidence_score: int = Field(description="A confidence score between 0 and 100 indicating the likelihood that this answer is correct.")
    explanation: str = Field(description="A brief explanation of how the clue leads to this answer.")

class Guesses(BaseModel):
    guesses: list[Guess] = Field(
        description="A list of five potential answers that fit the clue."
    )

In [22]:
llm = Ollama(
    model="qwen3.5:27b",
    request_timeout=1200.0,
    context_window=1000,
    temperature=0.1,
    json_mode=True,
)

sllm = llm.as_structured_llm(Guesses)

In [23]:
ordinal_map = {
    1: "first",
    2: "second",
    3: "third",
    4: "fourth",
    5: "fifth",
    6: "sixth",
    7: "seventh",
    8: "eighth",
    9: "ninth",
    10: "tenth",
}


def generate_prompt(clue: str, pattern: list[str]) -> str:
    length = len(pattern)
    pattern_str = "".join(
        [f"The {ordinal_map.get(i, f'{i}th')} letter is {letter if letter != '_' else 'unknown'},\n" for i, letter in enumerate(pattern, start=1)]
    )
    
    return f"""You are a crossword solver. You must give me five guesses that fit the clue and the exact letter pattern.

Constraints:
- Clue: {clue}
- Length: {length} letters

{pattern_str}

Final Output:
Return only the matching words in ALL CAPS, your confidence score (0-100), and an explanation. Each guess should be unique and should not contain spaces or punctuation.
"""

In [24]:
clue = "Black and white animal"
pattern = ["_", "_", "_", "_", "_"]

prompt = generate_prompt(clue, pattern)

print(prompt)

You are a crossword solver. You must give me five guesses that fit the clue and the exact letter pattern.

Constraints:
- Clue: Black and white animal
- Length: 5 letters

The first letter is unknown,
The second letter is unknown,
The third letter is unknown,
The fourth letter is unknown,
The fifth letter is unknown,


Final Output:
Return only the matching words in ALL CAPS, your confidence score (0-100), and an explanation. Each guess should be unique and should not contain spaces or punctuation.



In [ ]:
response = sllm.complete(prompt).raw
response.guesses.sort(key=lambda x: x.confidence_score, reverse=True)

In [ ]:
response.guesses

[Guess(answer='OTTER', confidence_score=80, explanation='Otters are known to have black and white fur.'),
 Guess(answer='SEALS', confidence_score=70, explanation='Seals are marine mammals with a distinctive black and white coloration.'),
 Guess(answer='PANDA', confidence_score=60, explanation='The giant panda is an endangered bear native to China that has distinct black and white markings.'),
 Guess(answer='SKUNK', confidence_score=50, explanation='Skunks are known for their ability to release a strong-smelling spray from their anal glands, but they also have distinctive black and white stripes.'),
 Guess(answer='RACCOON', confidence_score=40, explanation='Raccoons are mammals with a distinctive black and white mask on their face.')]